# Representing the aggregate book

The answers to Section 1 of the sample questions, `documentation/tex/questions`, run.
The questions concern Section 1.1 of the lecture notes and the book of Section 1.2.3.

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Prices are integer counts of ticks.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

## `question.signedStateToSides`: a level of size zero

In [ ]:
def to_sides(state):
    bids = {p: s for p, s in state.items() if s > 0}
    asks = {p: -s for p, s in state.items() if s < 0}
    return bids, asks

to_sides_a = to_sides

In [ ]:
def to_sides(state):
    bids = {p: s for p, s in state.items() if s > 0}
    asks = {p: -s for p, s in state.items() if p not in bids}
    return bids, asks

to_sides_b = to_sides

In [ ]:
def run(to_sides):
    state = {1000: 100, 999: 200, 998: 150, 1001: 0, 1002: -120, 1003: -180}
    bids, asks = to_sides(state)
    book = AggregateBook.from_levels(bids, asks)
    print(book.best_ask_price, book.spread)


a, b = outcome(run, to_sides_a), outcome(run, to_sides_b)
print("A:", a, "   B:", b)
verify("question.signedStateToSides", "C", [
    a == "1002 2" and b == "1002 2",
    a == "1002 2" and b == "1001 1",
    a == "1002 2" and b == "ValueError",
    a == "ValueError" and b == "ValueError",
])

## `question.imbalanceOverGridLevels`: grid levels against occupied levels

In [ ]:
def run():
    book = AggregateBook.from_levels(
        {1000: 100, 998: 300, 997: 200}, {1001: 150, 1002: 50, 1004: 400}
    )

    def imbalance(book, n):
        bids = sorted(book.bids.items(), reverse=True)[:n]
        asks = sorted(book.asks.items())[:n]
        b = sum(size for _, size in bids)
        a = sum(size for _, size in asks)
        return (b - a) / (b + a)

    print(round(book.queue_imbalance(GridDepth(2)), 3),
          round(imbalance(book, 2), 3))


printed = outcome(run)
print(printed)
verify("question.imbalanceOverGridLevels", "D", [
    printed == "-0.333 -0.333",
    printed == "0.333 0.333",
    printed == "0.333 -0.333",
    printed == "-0.333 0.333",
])

## `question.microPriceTowardsTheThinSide`: the micro-price against a size-weighted mid

In [ ]:
def run():
    def weighted_mid(book):
        bid, ask = book.best_bid_price, book.best_ask_price
        b, a = book.best_bid_size, book.best_ask_size
        return (bid * b + ask * a) / (b + a)

    book = AggregateBook.from_levels({1000: 300}, {1002: 100})
    print(book.micro_price, weighted_mid(book))


printed = outcome(run)
print(printed)
verify("question.microPriceTowardsTheThinSide", "A", [
    printed == "1001.5 1000.5",
    printed == "1000.5 1001.5",
    printed == "1001.5 1001.5",
    printed == "1001.0 1001.0",
])

## `question.floatPricesSplitALevel`: prices in currency, keyed by `int`

The stem states four facts of binary floating point; they are checked first.

In [ ]:
assert repr(10.04 / 0.01) == "1003.9999999999999"
assert repr(1004 * 0.01) == "10.040000000000001"
assert repr(1004 * 0.01 / 0.01) == "1004.0000000000001"
assert repr(10.03 / 0.01) == "1002.9999999999999"


def run():
    tick = 0.01
    orders = [(10.04, 100), (1004 * tick, 50), (10.03, 70)]
    levels = {}
    for price, size in orders:
        key = int(price / tick)
        levels[key] = levels.get(key, 0) + size
    print(levels)


printed = outcome(run)
print(printed)
verify("question.floatPricesSplitALevel", "B", [
    printed == "{1004: 150, 1003: 70}",
    printed == "{1003: 100, 1004: 50, 1002: 70}",
    printed == "{1003: 150, 1002: 70}",
    printed == "{1003: 170, 1004: 50}",
])

## `question.recorderAliasesTheSide`: a record of the side, and a copy of it

In [ ]:
def run():
    book = AggregateBook.from_levels({1000: 100}, {1002: 120})
    stream = [
        limit_order(1.0, 50, 1001, BUY),
        market_order(2.0, 30, SELL),
        withdrawal(3.0, 120, 1002, SELL),
    ]
    shared, copied = [], []
    for message in stream:
        book.apply(message, record=False)
        shared.append(book.levels_map(BUY))
        copied.append(dict(book.levels_map(BUY)))
    print(shared[0])
    print(copied[0])


first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.recorderAliasesTheSide", "C", [
    first == second == "{1000: 100, 1001: 50}",
    first == "{1000: 100}" and second == "{1000: 100, 1001: 50}",
    first == "{1000: 100, 1001: 20}" and second == "{1000: 100, 1001: 50}",
    first == second == "{1000: 100, 1001: 20}",
])

## `question.sizeIsNotVolume`: decreases of the best size against the traded size

In [ ]:
def run():
    book = AggregateBook.from_levels({1000: 100, 999: 200}, {1002: 120})
    stream = [
        market_order(1.0, 30, SELL),
        withdrawal(2.0, 50, 1000, BUY),
        limit_order(3.0, 40, 1000, BUY),
        market_order(4.0, 70, SELL),
    ]
    drops, traded = 0, 0
    before = book.best_bid_size
    for message in stream:
        book.apply(message, record=False)
        drops += max(before - book.best_bid_size, 0)
        before = book.best_bid_size
        traded += book.last_traded_size
    print(drops, traded)


printed = outcome(run)
print(printed)
verify("question.sizeIsNotVolume", "B", [
    printed == "100 100",
    printed == "80 100",
    printed == "140 100",
    printed == "80 80",
])

## `question.paddingSentinelsInTheSpread`: two sentinels of opposite sign

In [ ]:
def run():
    book = pd.DataFrame({
        "AskPrice1": [1002, 1003, 9999999999, 1002],
        "BidPrice1": [1000, -9999999999, 1001, 1001],
    })
    clean = book[book.BidPrice1 > 0]
    print(len(clean), (clean.AskPrice1 - clean.BidPrice1).mean())


printed = outcome(run)
print(printed)
rows, mean = printed.split()
rows, mean = int(rows), float(mean)
verify("question.paddingSentinelsInTheSpread", "D", [
    rows == 2 and mean == 1.5,
    rows == 3 and mean == 1.5,
    rows == 4 and abs(mean - 3.3e9) < 0.1e9,
    rows == 3 and abs(mean - 3.3e9) < 0.1e9,
])

## `question.levelsAreOneIndexed`: the $i$-th bid level, read two ways

In [ ]:
def run():
    book = AggregateBook.from_levels(
        {1000: 100, 998: 150, 997: 50}, {1002: 120, 1003: 180}
    )

    def size_a(book, i):
        return book.size_at(BUY, book.best_bid_price - (i - 1))

    def size_b(book, i):
        return book.size_at(BUY, book.best_bid_price - i)

    print([size_a(book, i) for i in (1, 2, 3)])
    print([size_b(book, i) for i in (1, 2, 3)])


first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.levelsAreOneIndexed", "A", [
    first == "[100, 0, 150]" and second == "[0, 150, 50]",
    first == "[100, 150, 50]" and second == "[150, 50, 0]",
    first == second == "[100, 0, 150]",
    first == "[0, 150, 50]" and second == "[100, 0, 150]",
])

## `question.sizeAtOnADictAgainstAList`: a dictionary against a scanned list

Each reading is timed on the same 2000 queries at each depth.
The choices are read off the growth of each time from $\ell = 10^3$ to $\ell = 10^4$,
and off their ratio at $\ell = 10^4$.

In [ ]:
def size_at_a(levels: dict[int, int], price: int) -> int:
    return levels.get(price, 0)

def size_at_b(levels: list[tuple[int, int]], price: int) -> int:
    for level_price, size in levels:
        if level_price == price:
            return size
    return 0

In [ ]:
def seconds_per_query(size_at, levels, queries):
    total = timeit.timeit(lambda: [size_at(levels, price) for price in queries], number=3)
    return total / 3 / len(queries)


rng = np.random.default_rng(1)
timings = {}
for depth in (1_000, 10_000):
    prices = [100_000 + 2 * k for k in range(depth)]
    sizes = rng.integers(1, 500, depth).tolist()
    queries = rng.choice(prices, 2_000).tolist()
    timings[depth] = (
        seconds_per_query(size_at_a, dict(zip(prices, sizes)), queries),
        seconds_per_query(size_at_b, sorted(zip(prices, sizes)), queries),
    )
    print(f"l = {depth:>6}:  A {timings[depth][0] * 1e9:8.1f} ns   "
          f"B {timings[depth][1] * 1e9:10.1f} ns per query")

growth_a = timings[10_000][0] / timings[1_000][0]
growth_b = timings[10_000][1] / timings[1_000][1]
b_over_a = timings[10_000][1] / timings[10_000][0]
print(f"growth of A {growth_a:.1f}x, growth of B {growth_b:.1f}x, B over A at 10^4 {b_over_a:.0f}x")
verify("question.sizeAtOnADictAgainstAList", "C", [
    b_over_a < 1,
    growth_a > 5 and growth_b > 5 and b_over_a > 1,
    b_over_a > 5 and growth_a < 2 and growth_b > 5,
    growth_a < 2 and growth_b < 2,
])